In [ ]:
%%bash
# # split votu reps into smaller chunks
# seqkit split2 \
#     uhvdb.votu_reps.fna.gz \
#     --by-size 20000 \
#     --out-dir votu_reps_split

# # create diamond db
# sbatch diamond_makedb.sh

# # run all pairwise combinations of blastp
# sbatch diamond_blastp.sh

In [ ]:
import polars as pl
# identify sequences not included in first aai clustering
old_reps = pl.read_csv('vclust/uhvdb_vclust_votu_reps.tsv', has_header=False)

final_reps = pl.read_csv('vclust/uhvdb_vclust_votu_reps_final.tsv', has_header=False)
final_not_in_old = final_reps.filter(~pl.col('column_1').is_in(old_reps['column_1']))
final_not_in_old.write_csv('final_reps_not_in_old.tsv', include_header=False)

# !seqkit grep \
#     ../figure_s5/uhvdb_hq_derep_hc_final.fna.gz \
#     --pattern-file final_reps_not_in_old.tsv \
#     --out-file votu_reps_split/uhvdb.votu_reps.part_012.fna.gz

In [12]:
import polars as pl
# group sequences by class
uhgv_ictv_taxonomy = (
    pl.read_csv('../figure_1/uhgv_metadata.tsv', separator='\t', columns=['uhgv_genome', 'uhgv_votu'])
        .join(
            pl.read_csv('../figure_3/votus_metadata.tsv', separator='\t', columns=['uhgv_votu', 'ictv_taxonomy', 'genome_length']),
            on='uhgv_votu', how='left'
        )
)

def extract_class(value):
    if value is None:
        return ""
    parts = value.split(';')
    if len(parts) > 1:
        for part in parts:
            if part.endswith('viricetes'):
                return part.strip()
            if part == 'Anelloviridae':
                return 'Cardeaviricetes'
    return ""


print(
    pl.read_csv('../uhvdb_clustering/vclust/uhvdb_vclust_votu_reps_final.tsv', separator='\t', new_columns=['seq_id'])
        .join(
            pl.read_csv('../figure_1/viruses.csvtk_concat.tsv', separator='\t', columns=['seq_name', 'taxonomy', 'contig_length', 'proviral_length']),
            left_on='seq_id', right_on='seq_name', how='left'
        )
        .join(
            pl.read_csv('../figure_1/uhvdb_final_metadata.tsv', separator='\t'),
            left_on='seq_id', right_on='seq_name', how='left'
        )
        .join(
            uhgv_ictv_taxonomy, left_on='seq_id', right_on='uhgv_genome', how='left'
        )
        .with_columns([
            pl.col('taxonomy').fill_null(pl.col('ictv_taxonomy')),
        ])
        .with_columns([
            pl.col('taxonomy').map_elements(extract_class, return_dtype=pl.String).alias('ictv_class')
        ])
        .unique('seq_id')
        .filter(pl.col('ictv_class') == "")
        .group_by('taxonomy').len()
        .write_csv('final_reps_no_class_count.tsv', separator='\t', include_header=True)
)

None


In [ ]:
%%bash
# # calculate self scores on self alignments
# sbatch diamond_selfscore.sh

# # calculate norm scores
# sbatch diamond_normscore.sh

In [ ]:
import polars as pl
import glob

final_reps_set = set(pl.read_csv('vclust/uhvdb_vclust_votu_reps_final.tsv', has_header=False)['column_1'])

# combine all norm score files
df_list = []
for file in glob.glob('/gscratch/scrubbed/carsonjm/diamond_blastp/votu_reps.part_*.diamond_normscore.tsv'):
    df = (
        pl.read_csv(file, separator='\t', columns=['query', 'reference', 'norm_score'])
            .filter(
                (pl.col('norm_score') >= 5.5) &
                (pl.col('query').is_in(final_reps_set)) &
                (pl.col('reference').is_in(final_reps_set))
            )
    )
    df_list.append(df)

all_norm_scores = pl.concat(df_list)

all_norm_scores.write_csv('protein_similarity/uhvdb_family_normscores.tsv', separator='\t', include_header=False)

In [ ]:
%%bash
# mcl \
#     protein_similarity/uhvdb_family_normscores.tsv \
#     --abc \
#     -sort revsize \
#     -te 12 \
#     -o protein_similarity/uhvdb_family.normscores.mcl


In [ ]:
import polars as pl

# read mcl clusters to a dictionary
mcl = (
    pl.read_csv('protein_similarity/uhvdb_family.normscores.mcl', has_header=False, row_index_name='cluster_id', row_index_offset=1)
)

mcl_clusters = {}

for row in mcl.iter_rows(named=True):
    for member in row['column_1'].split('\t'):
        mcl_clusters[member] = row['cluster_id']


# load subfamily pruned graph
subfamily_pruned = (
    pl.read_csv('../uhvdb_clustering/protein_similarity/uhvdb_family_normscores.tsv', separator='\t', has_header=False)
        .with_columns([
            pl.col('column_1').replace_strict(mcl_clusters, default='unassigned').alias('query_family'),
            pl.col('column_2').replace_strict(mcl_clusters, default='unassigned').alias('reference_family')
        ])
        .filter(
            (pl.col('query_family') == pl.col('reference_family')) &
            (pl.col('query_family') != 'unassigned') &
            (pl.col('column_3') >= 32)
        )
)


subfamily_pruned.write_csv('protein_similarity/uhvdb_subfamily_normscores.tsv', separator='\t', include_header=False)

!mcl \
    protein_similarity/uhvdb_subfamily_normscores.tsv \
    --abc \
    -sort revsize \
    -te 12 \
    -o protein_similarity/uhvdb_subfamily.normscores.mcl

In [ ]:
import polars as pl

# read mcl clusters to a dictionary
mcl = (
    pl.read_csv('protein_similarity/uhvdb_subfamily.normscores.mcl', has_header=False, row_index_name='cluster_id', row_index_offset=1)
)

mcl_clusters = {}

for row in mcl.iter_rows(named=True):
    for member in row['column_1'].split('\t'):
        mcl_clusters[member] = row['cluster_id']


# load genus pruned graph
genus_pruned = (
    pl.read_csv('../uhvdb_clustering/protein_similarity/uhvdb_subfamily_normscores.tsv', separator='\t', has_header=False)
        .with_columns([
            pl.col('column_1').replace_strict(mcl_clusters, default='unassigned').alias('query_subfamily'),
            pl.col('column_2').replace_strict(mcl_clusters, default='unassigned').alias('reference_subfamily')
        ])
        .filter(
            (pl.col('query_subfamily') == pl.col('reference_subfamily')) &
            (pl.col('query_subfamily') != 'unassigned') &
            (pl.col('column_3') >= 65)
        )
)


genus_pruned.write_csv('protein_similarity/uhvdb_genus_normscores.tsv', separator='\t', include_header=False)

!mcl \
    protein_similarity/uhvdb_genus_normscores.tsv \
    --abc \
    -sort revsize \
    -te 12 \
    -o protein_similarity/uhvdb_genus.normscores.mcl

In [ ]:
import polars as pl

# read mcl clusters to a dictionary
mcl = (
    pl.read_csv('protein_similarity/uhvdb_genus.normscores.mcl', has_header=False, row_index_name='cluster_id', row_index_offset=1)
)

mcl_clusters = {}

for row in mcl.iter_rows(named=True):
    for member in row['column_1'].split('\t'):
        mcl_clusters[member] = row['cluster_id']


# load subgenus pruned graph
subgenus_pruned = (
    pl.read_csv('../uhvdb_clustering/protein_similarity/uhvdb_genus_normscores.tsv', separator='\t', has_header=False)
        .with_columns([
            pl.col('column_1').replace_strict(mcl_clusters, default='unassigned').alias('query_genus'),
            pl.col('column_2').replace_strict(mcl_clusters, default='unassigned').alias('reference_genus')
        ])
        .filter(
            (pl.col('query_genus') == pl.col('reference_genus')) &
            (pl.col('query_genus') != 'unassigned') &
            (pl.col('column_3') >= 80)
        )
)


subgenus_pruned.write_csv('protein_similarity/uhvdb_subgenus_normscores.tsv', separator='\t', include_header=False)

!mcl \
    protein_similarity/uhvdb_subgenus_normscores.tsv \
    --abc \
    -sort revsize \
    -te 12 \
    -o protein_similarity/uhvdb_subgenus.normscores.mcl

In [1]:
# new uhgv votu selection script
import polars as pl

def load_mcl_clusters(mcl, unique):
    # assign sequences to mcl clusters
    clusters = {}

    cluster_id = 0
    with open(mcl, 'r') as mcl_file:
        for line in mcl_file:
            cluster_id += 1
            for node in line.strip().split():
                clusters[node] = cluster_id
    print("Number of clusters from MCL:", cluster_id)

    # assign unclustered sequences to their own cluster
    with open(unique, 'r') as unique_file:
        for line in unique_file:
            sequence = line.strip().split()[0]
            if sequence not in clusters:
                cluster_id += 1
                clusters[sequence] = cluster_id

    print("Number of total clusters:", cluster_id)

    return clusters

In [2]:
family_clusters = load_mcl_clusters('protein_similarity/uhvdb_family.normscores.mcl', 'vclust/uhvdb_vclust_votu_reps_final.tsv')
subfamily_clusters = load_mcl_clusters('protein_similarity/uhvdb_subfamily.normscores.mcl', 'vclust/uhvdb_vclust_votu_reps_final.tsv')
genus_clusters = load_mcl_clusters('protein_similarity/uhvdb_genus.normscores.mcl', 'vclust/uhvdb_vclust_votu_reps_final.tsv')
subgenus_clusters = load_mcl_clusters('protein_similarity/uhvdb_subgenus.normscores.mcl', 'vclust/uhvdb_vclust_votu_reps_final.tsv')

family_clusters_df = pl.DataFrame(list(family_clusters.items()), schema=["contig_id", "cluster_id"], orient='row')
family_clusters_df.write_csv('protein_similarity/uhvdb_family_clusters.tsv', separator='\t', include_header=True)

subfamily_clusters_df = pl.DataFrame(list(subfamily_clusters.items()), schema=["contig_id", "cluster_id"], orient='row')
subfamily_clusters_df.write_csv('protein_similarity/uhvdb_subfamily_clusters.tsv', separator='\t', include_header=True)

genus_clusters_df = pl.DataFrame(list(genus_clusters.items()), schema=["contig_id", "cluster_id"], orient='row')
genus_clusters_df.write_csv('protein_similarity/uhvdb_genus_clusters.tsv', separator='\t', include_header=True)

subgenus_clusters_df = pl.DataFrame(list(subgenus_clusters.items()), schema=["contig_id", "cluster_id"], orient='row')
subgenus_clusters_df.write_csv('protein_similarity/uhvdb_subgenus_clusters.tsv', separator='\t', include_header=True)

Number of clusters from MCL: 2160
Number of total clusters: 2357
Number of clusters from MCL: 8009
Number of total clusters: 16685
Number of clusters from MCL: 15220
Number of total clusters: 48841
Number of clusters from MCL: 12829
Number of total clusters: 104291
